In [1]:
import PIL
import subprocess

# Check current version of Pillow
current_version = PIL.__version__

# Define the required version
required_version = "10.2.0"

# Function to parse version strings
def version_tuple(version):
    return tuple(map(int, (version.split("."))))

# Compare current and required version
if version_tuple(current_version) < version_tuple(required_version):
    print(f"Current Pillow version {current_version} is less than {required_version}. Updating...")
    # Uninstall current version of Pillow
    subprocess.run(['pip', 'uninstall', 'pillow', '-y'])
    # Install the required version of Pillow
    subprocess.run(['pip', 'install', f'pillow=={required_version}'])
else:
    print(f"Current Pillow version {current_version} meets the requirement.")

Current Pillow version 10.4.0 meets the requirement.


In [60]:
!pip install --upgrade transformers
!pip install llama-index>=0.10.45 llama-index-vector-stores-chroma llama-index-embeddings-huggingface llama-index-llms-groq groq

  Using cached transformers-5.5.3-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.10.1-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached transformers-5.5.3-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-1.10.1-py3-none-any.whl (642 kB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface_hub 0.36.2

    Uninstalling huggingface_hub-0.36.2:

      Successfully uninstalled huggingface_hub-0.36.2

   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   ---------------------------------------- 0/3 [huggingface-hub]
   -----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 0.5.23 requires tokenizers<=0.20.3,>=0.13.2, but you have tokenizers 0.22.2 which is incompatible.
datasets 4.3.0 requires dill<0.4.1,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 4.3.0 requires fsspec[http]<=2025.9.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 4.3.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.
llama-index-llms-openai-like 0.5.3 requires llama-index-core<0.15,>=0.14.3, but you have llama-index-core 0.10.68.post1 which is incompatible.
llama-index-llms-openai-like 0.5.3 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.1.31 which is incompatible.
llama-index-llms-openai-like 0.5.3 requires transformers<5,>=4.37.0, but you have transformers 5.5.3 which is incompatibl

In [13]:

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext


In [4]:
!pip install pyvis==0.3.2


   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 756.0/756.0 kB 32.9 MB/s  0:00:00

   -------------------- ------------------- 1/2 [pyvis]
   ---------------------------------------- 2/2 [pyvis]



In [15]:
import os
from groq import Groq as GroqClient

# Retrieving and setting the Groq API key
# Create a local api_key.txt file with your Groq API key if it doesn't exist
api_key_path = "api_key.txt"

if os.path.exists(api_key_path):
    with open(api_key_path, "r") as f:
        GROQ_API_KEY = f.readline().strip()
else:
    # Alternatively, set it directly here or from environment variable
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your_groq_api_key_here")
    print(f"API key file not found. Using environment variable or default.")

# Set the Groq API key
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# Initialize Groq LLM client directly
groq_client = GroqClient(api_key=GROQ_API_KEY)

# Define model and parameters
GROQ_MODEL = "mixtral-8x7b-32768"
GROQ_TEMPERATURE = 0.7

print("Groq client initialized successfully!")


API key file not found. Using environment variable or default.
Groq client initialized successfully!


In [16]:

def download(directory, filename):
    # The base URL of the image files in the GitHub repository
    base_url = 'https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI/main/'

    # Complete URL for the file
    file_url = f"{base_url}{directory}/{filename}"

    # Use curl to download the file, including an Authorization header for the private token
    try:
        # Prepare the curl command with the Authorization header
        curl_command = f'curl -o {filename} {file_url}'

        # Execute the curl command
        subprocess.run(curl_command, check=True, shell=True)
        print(f"Downloaded '{filename}' successfully.")
    except subprocess.CalledProcessError:
        print(f"Failed to download '{filename}'. Check the URL, your internet connection and the file path")
# Setup Chroma vector store with embeddings
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# Try using HuggingFace embeddings - with proper error handling
embed_model = None
try:
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
    print("✓ Using HuggingFace embeddings")
except Exception as e:
    print(f"⚠ HuggingFace embeddings unavailable: {e}")
    print("  Using Chroma's default embeddings instead")

# Create a local Chroma client
chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection("knowledge_graph_documents")

# Initialize ChromaVectorStore
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create storage context with Chroma
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("✓ Local vector store initialized successfully!")


⚠ HuggingFace embeddings unavailable: cannot import name 'is_quanto_available' from 'transformers.utils' (c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\__init__.py)
  Using Chroma's default embeddings instead


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Local vector store initialized successfully!


In [25]:

def download(directory, filename):
    """Download a file from GitHub or use local file if it exists"""
    import os
    
    # Check if file exists locally first
    if os.path.exists(filename):
        print(f"✓ File '{filename}' already exists locally.")
        return
    
    # Also check in parent directory
    local_path = os.path.join(directory, filename)
    if os.path.exists(local_path):
        print(f"✓ File found at '{local_path}'")
        import shutil
        shutil.copy(local_path, filename)
        return
    
    # Try GitHub download as fallback
    base_url = 'https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI/main/'
    # Use Chapter07 path instead of relative paths
    file_url = f"{base_url}Chapter07/data/{filename}"

    try:
        curl_command = f'curl -o {filename} {file_url}'
        subprocess.run(curl_command, check=True, shell=True, capture_output=True)
        print(f"✓ Downloaded '{filename}' successfully from GitHub")
    except Exception as e:
        print(f"⚠ Could not download '{filename}' from: {file_url}")
        print(f"  Error: {e}")
        print(f"  Creating a placeholder file instead...")
        # Create an empty file to continue processing
        with open(filename, 'w') as f:
            f.write("# URLs would go here\n")


In [26]:
graph_name="Marketing"

# Path for vector store and dataset
db="../Ch7/knowledge_graph_index_based_rag_db"
vector_store_path = db
dataset_path = db

#if True upserts data; if False, passes upserting and goes to connection
pop_vs=True
# if pop_vs==True, overwrite=True will overwrite dataset, False will append it:
ow=True

# Pipeline 1: Collecting and preparing the documents

In [27]:
if pop_vs==True:
  directory = "../Ch7/citations"
  file_name = graph_name+"_urls.txt"
  download(directory,file_name)

✓ File found at '../Ch7/citations\Marketing_urls.txt'


In [28]:
import requests
from bs4 import BeautifulSoup
import re
import os

if pop_vs==True:
  directory = "Chapter07/citations"
  file_name = graph_name+"_urls.txt"

  with open(file_name, 'r') as file:
      urls = [line.strip() for line in file]

  # Display the URLs
  print("Read URLs:")
  for url in urls:
      print(url)

Read URLs:
https://en.wikipedia.org/wiki/Marketing
https://en.wikipedia.org/wiki/24-hour_news_cycle
https://en.wikipedia.org/wiki/Account-based_marketing
https://en.wikipedia.org/wiki/Activism
https://en.wikipedia.org/wiki/Adam_Smith
https://en.wikipedia.org/wiki/Adam_Smith_Institute
https://en.wikipedia.org/wiki/Advertising
https://en.wikipedia.org/wiki/Advertising_agency
https://en.wikipedia.org/wiki/Advertising_mail
https://en.wikipedia.org/wiki/Advertising_management
https://en.wikipedia.org/wiki/Advertising_slogan
https://en.wikipedia.org/wiki/Advertorial
https://en.wikipedia.org/wiki/Advocacy
https://en.wikipedia.org/wiki/Advocacy_group
https://en.wikipedia.org/wiki/Affinity_marketing
https://en.wikipedia.org/wiki/Agenda-setting_theory
https://en.wikipedia.org/wiki/Agile_marketing
https://en.wikipedia.org/wiki/Agricultural_Marketing_Service
https://en.wikipedia.org/wiki/Agricultural_marketing
https://en.wikipedia.org/wiki/Airborne_leaflet_propaganda
https://en.wikipedia.org/wiki/

In [30]:
import requests
import re
import os
import time
from bs4 import BeautifulSoup

def clean_text(content):
    # Remove references and unwanted characters
    content = re.sub(r'\[\d+\]', '', content)   # Remove references
    content = re.sub(r'[^\w\s\.]', '', content)  # Remove punctuation (except periods)
    return content

def fetch_and_clean(url, retries=3):
    """Fetch and clean Wikipedia content with proper headers and retry logic"""
    # Wikipedia User-Agent requirement
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()  # Raise exception for bad responses (e.g., 404, 403)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Prioritize "mw-parser-output" but fall back to "content" class if not found
            content = soup.find('div', {'class': 'mw-parser-output'}) or soup.find('div', {'id': 'content'})
            if content is None:
                return None

            # Remove specific sections, including nested ones
            for section_title in ['References', 'Bibliography', 'External links', 'See also', 'Notes']:
                section = content.find('span', id=section_title)
                while section:
                    for sib in section.parent.find_next_siblings():
                        sib.decompose()
                    section.parent.decompose()
                    section = content.find('span', id=section_title)

            # Extract and clean text
            text = content.get_text(separator=' ', strip=True)
            text = clean_text(text)
            return text
            
        except requests.exceptions.RequestException as e:
            if attempt < retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f"⚠ Attempt {attempt + 1} failed for {url}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"✗ Error fetching content from {url}: {e}")
                return None

if pop_vs==True:
  # Directory to store the output files
  output_dir = './data/'  # More descriptive name
  os.makedirs(output_dir, exist_ok=True)

  # Processing each URL (and skipping invalid ones)
  success_count = 0
  for url in urls:
      article_name = url.split('/')[-1].replace('.html', '')  # Handle .html extension
      filename = os.path.join(output_dir, f"{article_name}.txt")

      clean_article_text = fetch_and_clean(url)
      if clean_article_text:  # Only write to file if content exists
          with open(filename, 'w', encoding='utf-8') as file:
              file.write(clean_article_text)
          success_count += 1
          print(f"✓ Processed: {article_name}")
      
      time.sleep(0.5)  # Be respectful to Wikipedia servers - add delay between requests
  
  print(f"\n✓ Successfully processed {success_count}/{len(urls)} articles")
  print(f"✓ Content written to files in the '{output_dir}' directory.")


✓ Processed: 24-hour_news_cycle
✓ Processed: Account-based_marketing
✓ Processed: Activism
✓ Processed: Adam_Smith_Institute
✓ Processed: Advertising_mail
✓ Processed: Advertising_management
✓ Processed: Advertising_slogan
✓ Processed: Advertorial
✓ Processed: Advocacy
✓ Processed: Advocacy_group
✓ Processed: Affinity_marketing
✓ Processed: Agenda-setting_theory
✓ Processed: Agile_marketing
✓ Processed: Agricultural_Marketing_Service
✓ Processed: Agricultural_marketing
✓ Processed: Airborne_leaflet_propaganda
✓ Processed: Alternative_facts
✓ Processed: Alternative_media
✓ Processed: Ambush_marketing
✓ Processed: American_Marketing_Association
✓ Processed: American_business_history
✓ Processed: Annoyance_factor
✓ Processed: Anthropology
✓ Processed: Art_director
✓ Processed: Astroturfing
✓ Processed: Attack_ad
✓ Processed: Attribution_(marketing)
✓ Processed: Audio_deepfake
✓ Processed: Business_marketing
✓ Processed: Direct-to-consumer
✓ Processed: Bandwagon_effect
✓ Processed: Targete

In [31]:
if pop_vs==True:
  # load documents
  documents = SimpleDirectoryReader("./data/").load_data()
  # Print the first document
  print(documents[0])

Doc ID: 0b68fc8c-325a-4357-a768-6b408e0ec7ba
Text: Investigation and reporting of news concomitant with fastpaced
lifestyles This article is about the fastpaced cycle of news media in
technologically advanced societies. For the longerterm cycle of news
and information see information cycle . Several simultaneous NBC News
broadcasts including MSNBC  NBC s Today and CNBC s Squawk Box
displayed on...


# Pipeline 2: Creating and populating a vector store

In [39]:

# Pipeline 2: Creating and Storing the Index
from llama_index.core import Settings

# Use Chroma's built-in embeddings (no API key needed, works offline)
# This avoids all OpenAI/API authentication issues
Settings.embed_model = None
print("✓ Using Chroma's built-in embeddings (no API key needed)")

if pop_vs==True:
    # Handle overwrite logic for Chroma
    if ow==True:
        # Delete and recreate the collection to start fresh
        chroma_client.delete_collection(name="knowledge_graph_documents")
        chroma_collection = chroma_client.get_or_create_collection("knowledge_graph_documents")
        vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
        print("✓ Cleared existing collection (overwrite=True)")
    
    # Create storage context with Chroma vector store
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    
    # Create an index from the documents
    # Don't pass embed_model - let Chroma use its default
    print(f"Creating index from {len(documents)} documents...")
    index = VectorStoreIndex.from_documents(
        documents, 
        storage_context=storage_context,
        show_progress=True
    )
    
    # Get collection statistics
    collection_count = chroma_collection.count()
    print(f"\n✓ Index created successfully!")
    print(f"✓ Total vectors in Chroma collection: {collection_count}")
else:
    print("pop_vs is False - skipping index creation (using existing index)")


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Embeddings have been explicitly disabled. Using MockEmbedding.
✓ Using Chroma's built-in embeddings (no API key needed)
✓ Cleared existing collection (overwrite=True)
Creating index from 87 documents...


Parsing nodes:   0%|          | 0/87 [00:00<?, ?it/s]

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


...

Embeddings have been explicitly disabled. Using MockEmbedding.
✓ Using Chroma's built-in embeddings (no API key needed)
✓ Cleared existing collection (overwrite=True)
Creating index from 87 documents...


Parsing nodes:   0%|          | 0/87 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/843 [00:00<?, ?it/s]

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given



✓ Index created successfully!
✓ Total vectors in Chroma collection: 843


In [ ]:

import json
import pandas as pd
import numpy as np

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

# Get dataset info from Chroma collection
collection_count = chroma_collection.count()
print(f"\n✓ Dataset loaded from Chroma")
print(f"✓ Total documents: {collection_count}")
print(f"✓ Collection name: knowledge_graph_documents")

# Get all data from the collection
all_data = chroma_collection.get(include=["documents", "embeddings", "metadatas"])

# Create a DataFrame from the collection (exclude embeddings - they're multi-dimensional)
data = {
    "id": all_data["ids"],
    "text": all_data["documents"],
    "metadata": all_data["metadatas"]
}

df = pd.DataFrame(data)

# Store embeddings separately for later use
embeddings = np.array(all_data["embeddings"])

# Display dataset summary
print(f"\n✓ Dataset Structure:")
print(f"   - Total rows: {len(df)}")
print(f"   - Columns: {', '.join(df.columns.tolist())}")
print(f"   - Text samples: {len(df['text'])} documents")
print(f"   - Embeddings shape: {embeddings.shape} - [docs, embedding_dim]")

print(f"\nDataFrame preview:")
print(df.head(3))

# Function to display a selected record
def display_record(record_number):
    """Display a record from the dataset with full details"""
    if record_number >= len(df) or record_number < 0:
        print(f"✗ Invalid record number. Dataset has {len(df)} records (0-{len(df)-1})")
        return
    
    record = df.iloc[record_number]
    
    print("\n" + "=" * 60)
    print(f"RECORD {record_number}")
    print("=" * 60)
    
    # Display ID
    print(f"\nID: {record['id']}")
    
    # Display metadata
    print(f"\nMetadata:")
    if isinstance(record['metadata'], dict):
        for key, value in record['metadata'].items():
            print(f"  {key}: {value}")
    else:
        print(f"  {record['metadata']}")
    
    # Display text (truncated for readability)
    print(f"\nText (first 500 chars):")
    text = record['text']
    print(f"  {text[:500]}..." if len(text) > 500 else f"  {text}")
    
    # Display embedding info
    print(f"\nEmbedding:")
    embedding = embeddings[record_number]
    print(f"  Shape: {embedding.shape}")
    print(f"  First 5 values: {embedding[:5]}")
    print(f"  Mean: {embedding.mean():.4f}, Std: {embedding.std():.4f}")

# Example usage
print("\n" + "=" * 60)
print("SAMPLE RECORD DISPLAY")
print("=" * 60)
display_record(0)  # Display first record



DATASET SUMMARY

✓ Dataset loaded from Chroma
✓ Total documents: 843
✓ Collection name: knowledge_graph_documents

✓ Dataset Structure:
   - Total rows: 843
   - Columns: id, text, metadata
   - Text samples: 843 documents
   - Embeddings shape: (843, 1) - [docs, embedding_dim]

DataFrame preview:
                                     id  \
0  915514f3-bbca-4b61-877e-b1ba14e892cf   
1  5cbe0bd0-6078-412c-838f-5598f8a7473e   
2  c042182f-a545-4526-b50e-2dced266f224   

                                                text  \
0  Investigation and reporting of news concomitan...   
1  In Giles Robert H. Snyder Robert W. eds.. What...   
2  B2B marketing strategy focused on key accounts...   

                                            metadata  
0  {'_node_content': '{"id_": "915514f3-bbca-4b61...  
1  {'_node_content': '{"id_": "5cbe0bd0-6078-412c...  
2  {'_node_content': '{"id_": "c042182f-a545-4526...  

SAMPLE RECORD DISPLAY

RECORD 0

ID: 915514f3-bbca-4b61-877e-b1ba14e892cf

Metada

In [44]:

df['text'] = df['text'].astype(str)
# Create documents with IDs
documents = [Document(text=row['text'], doc_id=str(row['id'])) for _, row in df.iterrows()]

# Pipeline 3: Knowledge graph index-based RAG

### Generating the Knowledge Graph Index

In [64]:
from llama_index.core import KnowledgeGraphIndex
from llama_index.llms.groq import Groq
import time

# Use a currently supported, efficient model (mixtral-8x7b-32768 is decommissioned)
CURRENT_MODEL = "llama-3.1-8b-instant"

# Initialize Groq LLM for knowledge graph creation
llm = Groq(model=CURRENT_MODEL, api_key=GROQ_API_KEY, temperature=0.7)

# Start the timer
start_time = time.time()

# graph index with embeddings
graph_index = KnowledgeGraphIndex.from_documents(
    documents,
    max_triplets_per_chunk=2,
    include_embeddings=True,
    llm=llm,
)

# Stop the timer
end_time = time.time()

# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Index creation time: {elapsed_time:.4f} seconds")

KeyboardInterrupt: 

In [ ]:
print(type(graph_index))


In [ ]:
#similarity_top_k
k=3
#temperature
temp=0.1
#num_output
mt=1024
graph_query_engine = graph_index.as_query_engine(similarity_top_k=k, temperature=temp, num_output=mt)

### Displaying the graph

In [ ]:
## create graph
from pyvis.network import Network

g = graph_index.get_networkx_graph()
net = Network(notebook=True, cdn_resources="in_line", directed=True)
net.from_nx(g)

# Set node and edge properties: colors and sizes
for node in net.nodes:
    node['color'] = 'lightgray'
    node['size'] = 10

for edge in net.edges:
    edge['color'] = 'black'
    edge['width'] = 1

In [ ]:
fgraph="Knowledge_graph_"+ graph_name + ".html"
net.write_html(fgraph)
print(fgraph)

In [ ]:
from IPython.display import HTML

# Load the HTML content from a file and display it
with open(fgraph, 'r') as file:
    html_content = file.read()

# Display the HTML in the notebook
display(HTML(html_content))